## 1) Setup

In [12]:

# If needed, install once:
# %pip install --upgrade sentence-transformers numpy pandas tqdm
# For local LLM via Ollama:
# %pip install --upgrade requests
# Vector store
# %pip install --upgrade chromadb

### PreDev Setup

In [13]:
from importlib import reload  # Reload modules during development
import os  # OS utilities
import requests  # HTTP requests
import numpy as np  # Numerical operations
import faiss  # Vector similarity search

import database  # Local database module
from database import AmberChromaAPI  # Amber-Chroma interface

from pypdf import PdfReader  # PDF reading
from sentence_transformers import SentenceTransformer  # Text embeddings

reload(database)  # Refresh module changes


<module 'database' from '/home/bsauce11/RAG_Prototype/Code_Saucedo/My_PreDev/database.py'>

In [14]:
## langchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.documents import Document

## vectorstores
from langchain_community.vectorstores import Chroma

## utility imports
import numpy as np
from typing import List

### Document Splitting

In [15]:
# # Initialize text splitter
# text_splitter = RecursiveCharacterTextSplitter(
#     chunk_size=500,  # Maximum size of each chunk
#     chunk_overlap=50,  # Overlap between chunks to maintain context
#     length_function=len,
#     separators=[" "]  # Hierarchy of separators
# )
# chunks=text_splitter.split_documents(documents)
#
# print(f"Created {len(chunks)} chunks from {len(documents)} documents")
# print(f"\nChunk example:")
# print(f"Content: {chunks[0].page_content[:150]}...")
# print(f"Metadata: {chunks[0].metadata}")

In [16]:
# chunks

### Embedding Models

In [17]:
### Huggingface model

from langchain_huggingface import HuggingFaceEmbeddings

## Initialize a simple Embedding model(no API Key needed!)
embeddings=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
embeddings

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [18]:
# vector=embeddings.embed_query(sample_text)
# vector

### Amber ChromaDB

In [19]:
# Chroma DB instance
API_CHROMA_DB = AmberChromaAPI(db_path="/opt/chromadb/data/prompt_db")
# Embedding model
EMBEDDER = SentenceTransformer("all-MiniLM-L6-v2")
# PDF file path
PDF_ADDRESS = "Amber25.pdf"
# Local Ollama server URL
OLLAMA_URL = "http://127.0.0.1:11434"


Using local ChromaDB path: /opt/chromadb/data/prompt_db


In [20]:
threshold_ChromaDB = 0.35 # Similarity threshold for Mails
threshold_PDF = 0.45  # Similarity threshold for PDF results

### Chunking

In [21]:
# # Retrieve relevant chunks from hybrid retriever
# chunks = retrieve_with_pdf(
#     question,
#     k_chroma=50,     # Number of ChromaDB results
#     k_pdf=5,         # Number of PDF results
#     threshold=threshold_ChromaDB    # Similarity threshold
# )

In [22]:
import chromadb
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# If you already have this client, reuse it:
client = chromadb.PersistentClient(path="/opt/chromadb/data/prompt_db")

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    client=client,
    collection_name="rag_collection",
    embedding_function=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

/tmp/ipykernel_3515886/593024963.py:10: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


In [23]:
# ## Create a Chromdb vector store
# persist_directory="/opt/chromadb/data/prompt_db"
#
# ## Initialize Chromadb with HuggingFace embeddings
# vectorstore=Chroma.from_documents(
#     documents=chunks,
#     embedding=HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"),
#     persist_directory=persist_directory,
#     collection_name="rag_collection"
#
# )
#
print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Vector store name: {vectorstore._collection.name}")

Vector store created with 0 vectors
Vector store name: rag_collection


### Test Similarity Search

In [24]:
query="What is Amber?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[]

In [25]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: What is Amber?

Top 0 similar chunks:


### Advanced Similarity Search With Scores

In [26]:
results_scores=vectorstore.similarity_search_with_score(query,k=3)
results_scores

[]

#### Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the distance metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

- Higher scores = MORE similar
- Range: -1 to 1 (1 being identical)

#### Initialize LLM, RAG Chain, Prompt Template,Query the RAG system

In [27]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1:8b",
    temperature=0
)
llm

ChatOllama(model='llama3.1:8b', temperature=0.0)

In [28]:
# from langchain_community.llms import Ollama
#
# #OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
# #OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")
#
# llm = Ollama(model="llama3.1:8b")

In [29]:
# Use the LLM to generate prompt
test_response=llm.invoke("What is Amber?")
test_response

AIMessage(content='Amber is a fascinating substance with a rich history. Here\'s what it is:\n\n**Definition:** Amber is a fossilized tree resin that has been hardened over time, often containing ancient plant and animal remains.\n\n**Formation:** Amber forms when pine trees or other conifers produce sticky resin to protect themselves from insects, diseases, and environmental stressors. This resin can seep out of the tree\'s bark and harden in place, trapping small organisms like insects, spiders, and even tiny mammals within its sticky matrix.\n\n**Properties:** Amber is a translucent, yellowish-brown or golden-colored substance with a waxy texture. It has a distinctive "glow" due to the way it refracts light. Amber can be found in various forms, including:\n\n1. **Fossilized resin**: The original tree resin that has hardened over time.\n2. **Amberite**: A type of amber that contains more than 10% of organic matter, such as plant or animal remains.\n3. **Baltic amber**: A specific typ

### Create RAG Chain Alternative - Using LCEL (LangChain Expression Language)

In [30]:
# ## Convert vector store to retriever
# retriever=vectorstore.as_retriever(
#      search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
#  )
# retriever

In [31]:
# Even more flexible approach using LCEL
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

In [32]:
SYSTEM_PROMPT = """
You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).


CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention an Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.
7) Do NOT include citation markers, chunk labels, similarity scores,
   reference numbers, or metadata in your output.
8) Do NOT repeat metadata from the Context.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output only the final answer and
Include citations, references, or metadata.
"""

In [33]:
"""
You are a concise, technical assistant for Amber molecular simulation users. "
    Answer the user's question using ONLY the provided context.
    If the answer cannot be determined from the context, say you do not know.
    Cite sources using [Title#chunkN] notation.
    Do not speculate or introduce external knowledge.
"""

'\nYou are a concise, technical assistant for Amber molecular simulation users. "\n    Answer the user\'s question using ONLY the provided context.\n    If the answer cannot be determined from the context, say you do not know.\n    Cite sources using [Title#chunkN] notation.\n    Do not speculate or introduce external knowledge.\n'

In [34]:
from langchain_core.prompts import ChatPromptTemplate

# Create a custom prompt
custom_prompt = ChatPromptTemplate.from_template("""You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite
and AmberTools workflows.

You answer questions strictly using the provided Context (archive discussions and manuals).


CORE RULES-
1) Use ONLY the provided Context to generate your answer.
2) Do NOT use outside knowledge or prior training information.
3) You may logically reason based on information in the Context,
   but do NOT introduce new facts that are not supported by it.
4) If the Context contains relevant information, use it to answer as completely as possible.
5) Do NOT mention an Persona, Identity, or Role in your answer.
6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.
7) Do NOT include citation markers, chunk labels, similarity scores,
   reference numbers, or metadata in your output.
8) Do NOT repeat metadata from the Context.

STYLE-
- Start with a clear, direct answer.
- Then provide a concise technical explanation.
- Include practical AMBER-specific guidance only if supported by the Context.
- Address multiple sub-questions in the same order asked.
- Be precise, professional, and focused.
- Avoid unnecessary verbosity.

Output only the final answer.
Do not include citations, references, or metadata.

Context:
{context}

Question: {question}

Answer:""")
custom_prompt

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite\nand AmberTools workflows.\n\nYou answer questions strictly using the provided Context (archive discussions and manuals).\n\n\nCORE RULES-\n1) Use ONLY the provided Context to generate your answer.\n2) Do NOT use outside knowledge or prior training information.\n3) You may logically reason based on information in the Context,\n   but do NOT introduce new facts that are not supported by it.\n4) If the Context contains relevant information, use it to answer as completely as possible.\n5) Do NOT mention an Persona, Identity, or Role in your answer.\n6) Do NOT fabricate AMBER commands, flags, filenames, or parameter values.\n7) Do NOT include citation marker

In [35]:
## Convert vector store to retriever
retriever=vectorstore.as_retriever(
     search_kwarg={"k":3} ## Retrieve top 3 relevant chunks
 )

retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x147197771640>, search_kwargs={})

In [36]:
## Format the output documents for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [37]:
## Build the chain ussing LCEL

rag_chain_lcel=(
    {
        "context":retriever | format_docs,
        "question": RunnablePassthrough()
     }
    | custom_prompt
    | llm
    | StrOutputParser()
)

rag_chain_lcel

{
  context: VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x147197771640>, search_kwargs={})
           | RunnableLambda(format_docs),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='You are AmberRAG, an expert AI assistant specializing in the AMBER molecular dynamics suite\nand AmberTools workflows.\n\nYou answer questions strictly using the provided Context (archive discussions and manuals).\n\n\nCORE RULES-\n1) Use ONLY the provided Context to generate your answer.\n2) Do NOT use outside knowledge or prior training information.\n3) You may logically reason based on information in the Context,\n   but do NOT introduce new facts that are not supported by it.\n4) If t

In [38]:
response=rag_chain_lcel.invoke("What is Amber")
response

'The AMBER (Assisted Model Building with Energy Refinement) molecular dynamics suite is a software package used for simulating the behavior of molecules in various environments. It was originally developed by Peter Kollman and his group at the University of California, San Francisco.\n\nTechnical Explanation:\nAMBER uses a combination of classical mechanics and quantum mechanics to simulate the interactions between atoms and molecules. The software includes tools for building molecular models, calculating energies, and performing simulations using various algorithms such as molecular dynamics (MD) and Monte Carlo (MC). AMBER also includes a parameter set that is widely used in biomolecular simulations.\n\nPractical Guidance:\nTo use AMBER, one typically starts by preparing the input files, including the topology file (.prmtop) and the coordinate file (.crd or .pdb), which contain information about the molecular structure. The user then runs the simulation using a command such as "sande

In [39]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("What is Amber")


In [40]:
#retriever.get_relevant_documents("What is Deep Learning")
retDocs = retriever.invoke("What is Deep Learning")
retDocs

[]

In [41]:
# Query using the LCEL approach - Fixed version
def query_rag_lcel(question):
    print(f"Question: {question}")
    print("-" * 50)

    # Method 1: Pass string directly (when using RunnablePassthrough)
    answer = rag_chain_lcel.invoke(question)
    print(f"Answer: {answer}")

    # Get source documents separately if needed
    docs = retriever.invoke(question)
    #docs = retriever.get_relevant_documents(question)
    print("\nSource Documents:")
    for i, doc in enumerate(docs):
        print(f"\n--- Source {i+1} ---")
        print(doc.page_content[:200] + "...")

In [42]:
# Test LCEL chain
print("Testing LCEL Chain:")
query_rag_lcel("What can I use Amber for?")

Testing LCEL Chain:
Question: What can I use Amber for?
--------------------------------------------------
Answer: Molecular dynamics simulations, energy minimization, normal mode analysis, free energy calculations, molecular mechanics, and Monte Carlo simulations.

The AMBER suite is a collection of programs that can be used to perform various types of molecular dynamics simulations. These include energy minimization, normal mode analysis, and free energy calculations. The suite also includes tools for preparing molecular structures, such as antechamber and tleap, which can be used to generate force field parameters and topology files.

The AMBER suite is commonly used in a variety of fields, including chemistry, biology, and physics. It has been applied to the study of protein-ligand interactions, protein folding, and molecular recognition, among other topics.

Source Documents:


In [43]:
query_rag_lcel("Why should SHAKE be disabled during minimization in AMBER?")

Question: Why should SHAKE be disabled during minimization in AMBER?
--------------------------------------------------
Answer: During minimization in AMBER, SHAKE should be disabled because it can lead to an artificial stabilization of the system. SHAKE is a harmonic constraint that fixes all bonds involving hydrogen atoms, which can prevent the system from relaxing and reaching its true minimum energy state.

Technical explanation:
The SHAKE algorithm is used to constrain bond lengths involving hydrogen atoms, which are typically much lighter than other atoms in the molecule. This can lead to an artificial stabilization of the system, as the hydrogen atoms are not allowed to move freely. During minimization, it's essential to allow the system to relax and reach its true minimum energy state, rather than being artificially stabilized by SHAKE.

Practical guidance:
To disable SHAKE during minimization in AMBER, use the following command: `sander -p input.parm -o output.trr -c input.crd

In [44]:
query_rag_lcel("How does AMBER handle time-series anomaly detection?")

Question: How does AMBER handle time-series anomaly detection?
--------------------------------------------------
Answer: AMBER uses the "ptraj" module to analyze trajectory files and detect anomalies in molecular dynamics simulations. ptraj can be used with various analysis tools such as RMSD, RMSF, and others to identify unusual behavior.

Technical Explanation:
The ptraj module reads in a trajectory file generated by AMBER's pmemd or sander engines and allows users to perform various analyses on the data. Users can specify which atoms to analyze, what type of analysis to perform (e.g., RMSD, RMSF), and how to filter the results.

Practical Guidance:
To detect anomalies using ptraj, users should first generate a trajectory file from their simulation using pmemd or sander. Then, they can use ptraj to analyze the trajectory and identify unusual behavior by specifying relevant analysis tools and filters. For example:

ptraj input.pdb output.trr :GO 1-10 RMSD

This command would calculat

In [45]:
query_rag_lcel("How can I get SHAKE to consider two different residue names to be water?")

Question: How can I get SHAKE to consider two different residue names to be water?
--------------------------------------------------
Answer: To get SHAKE to consider two different residue names as water, you need to assign them a common atom type that is associated with water in the AMBER force field. 

In the AMBER parameter file (par_all36_prot.dat or par_all36mio_prot.dat), look for the "water" atom types and their corresponding residue names. Then, modify your topology file to assign the desired residue names to these water atom types.

For example, if you want residues 'HOH' and 'WAT' to be treated as water, you would replace their atom types with those of the water molecule in the parameter file. This will allow SHAKE to apply its constraints to these residues as well.

Note that this approach assumes a standard AMBER force field and topology files. If you are using a custom or modified force field, you may need to adjust your parameter and topology files accordingly.

Source Do

In [46]:
query_rag_lcel("How can I add the CG protein into a CG bilayer?")

Question: How can I add the CG protein into a CG bilayer?
--------------------------------------------------
Answer: To add the CG protein into a CG bilayer, you can use the `tleap` module in AMBERTools to create the system and then run a simulation. 

First, create a new input file for `tleap`, e.g., `system.tleap`. In this file, define the CG protein and the CG bilayer using their respective parameters. Then, use the `:load` command to load the pre-equilibrated CG bilayer from an external file or generate it on-the-fly.

For example:
```
source leaprc.ff14SB
protein = loadpdb protein.pdb
bilayer = loadcgbilayer

:load cg_bilayer.cgb
```
Next, run `tleap` to create the system and save it in a new file, e.g., `system.prmtop`. This will generate the topology and coordinate files for your CG protein-bilayer system.

Finally, use the `sander` module to run a simulation on this system. Make sure to specify the correct force field and parameters for the CG model.

Note: The specific command

In [47]:
query_rag_lcel("How do I obtain a Z-DNA structure from NAB?")

Question: How do I obtain a Z-DNA structure from NAB?
--------------------------------------------------
Answer: To obtain a Z-DNA structure from NAB, you can use the `sander` command with the `-ZDNA` flag. This will convert the DNA structure to its corresponding Z-DNA conformation.

Technical explanation:
The `-ZDNA` flag in `sander` uses the Z-DNA conversion algorithm developed by Kabsch and Sander (1988) to transform a B-DNA structure into its Z-DNA equivalent. This is achieved by adjusting the sugar pucker, base pair step parameters, and other structural features of the DNA molecule.

Practical guidance:
To use this flag, simply add `-ZDNA` to your `sander` command line, along with any other relevant flags or options required for your simulation. For example: `sander -i input.inp -o output.out -ZDNA`.

Source Documents:


In [48]:
query_rag_lcel("If we do not neutralize our system of protein-ligand complex properly or the addition of ions is not appropriate, will the properties like planarity of the system be affected?")

Question: If we do not neutralize our system of protein-ligand complex properly or the addition of ions is not appropriate, will the properties like planarity of the system be affected?
--------------------------------------------------
Answer: Yes, if the system is not properly neutralized or ions are not added appropriately, it can affect the properties of the system, including planarity.

The planarity of a system in AMBER simulations depends on the accurate representation of electrostatic interactions. If the system is not properly neutralized, the electrostatic potential may be distorted, leading to deviations from planarity. Similarly, if ions are not added appropriately, it can alter the distribution of charges within the system, also affecting planarity.

To maintain planarity and ensure accurate simulations, it is essential to carefully set up the system's charge neutrality and ion composition according to the specific requirements of your simulation. This typically involves a

### Advanced Rag Techniques- Conversational Memory
Understanding Conversational Memory in RAG
Conversational memory enables RAG systems to maintain context across multiple interactions. This is crucial for:

Follow-up questions that reference previous answers
Pronoun resolution (e.g., "it", "they", "that")
Context-dependent queries that build on prior discussion
Natural dialogue flow where users don't repeat context

Key Challenge:
Traditional RAG retrieves documents based only on the current query, missing important context from the conversation. For example:

User: "Tell me about Python"
Bot: explains Python programming language
User: "What are its main libraries?" ← "its" refers to Python, but retriever doesn't know this

Solution:
The modern approach uses a two-step process:

Query Reformulation: Transform context-dependent questions into standalone queries
Context-Aware Retrieval: Use the reformulated query to fetch relevant documents

- create_history_aware_retriever: Makes the retriever understand conversation context
- MessagesPlaceholder: Placeholder for chat history in prompts
- HumanMessage/AIMessage: Structured message types for conversation history

In [49]:
# from langchain_classic.chains.history_aware_retriever import create_history_aware_retriever
# from langchain_core.prompts import MessagesPlaceholder
# from langchain_core.messages import HumanMessage, AIMessage

In [50]:
# ## create a prompt that includes the chat history
# contextualize_q_system_prompt = """Given a chat history and the latest user question
# which might reference context in the chat history, formulate a standalone question
# which can be understood without the chat history. Do NOT answer the question,
# just reformulate it if needed and otherwise return it as is."""
#
# contextualize_q_prompt = ChatPromptTemplate.from_messages([
#     ("system", contextualize_q_system_prompt),
#     MessagesPlaceholder("chat_history"),
#     ("human", "{input}"),
# ])

In [51]:
# ## create history aware retriever
# history_aware_retriever = create_history_aware_retriever(
#     llm, retriever, contextualize_q_prompt
# )
# history_aware_retriever

In [52]:
# from langchain_classic.chains.retrieval import create_retrieval_chain
# from langchain_classic.chains.combine_documents import create_stuff_documents_chain
#
# # Create a new document chain with history
# qa_system_prompt = """You are an assistant for question-answering tasks.
# Use the following pieces of retrieved context to answer the question.
# If you don't know the answer, just say that you don't know.
# Use three sentences maximum and keep the answer concise.
#
# Context: {context}"""
#
# qa_prompt = ChatPromptTemplate.from_messages([
#     ("system", qa_system_prompt),
#     MessagesPlaceholder("chat_history"),
#     ("human", "{input}"),
# ])
#
# question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
#
# # Create conversational RAG chain
# conversational_rag_chain = create_retrieval_chain(
#     history_aware_retriever,
#     question_answer_chain
# )
# print("Conversational RAG chain created!")

In [53]:
# chat_history=[]
# # First question
# result1 = conversational_rag_chain.invoke({
#     "chat_history": chat_history,
#     "input": "What is machine learning?"
# })
# print(f"Q: What is machine learning?")
# print(f"A: {result1['answer']}")

In [54]:
# chat_history.extend([
#     HumanMessage(content="What is machine learning"),
#     AIMessage(content=result1['answer'])
# ])

In [55]:
# chat_history

In [56]:
# ## Follow up question
# # Follow-up question
# result2 = conversational_rag_chain.invoke({
#     "chat_history": chat_history,
#     "input": "What are its main types?"  # Refers to ML from previous question
# })
# result2

In [57]:
# result2['answer']